In [1]:
!pip install qiskit qiskit-aer qiskit-nature qiskit-algorithms pyscf matplotlib pandas --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 59.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 88.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 15.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.9 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
from pyscf import gto, scf, mcscf, cc

HARTREE_TO_EV = 27.211386245988
EXPERIMENTAL_PI_PISTAR_EV = 7.6
np.random.seed(1)

In [3]:
def rotate_z(xyz, angle_deg):
    a = np.radians(angle_deg)
    R = np.array([[np.cos(a), -np.sin(a), 0],
                  [np.sin(a),  np.cos(a), 0],
                  [0, 0, 1]])
    return R @ np.array(xyz)

def build_geometry(twist_deg: float = 0.0) -> str:
    atoms = {k: v.copy() for k, v in equilibrium_atoms.items()}
    if twist_deg != 0.0:
        atoms["H3"] = rotate_z(atoms["H3"], twist_deg)
        atoms["H4"] = rotate_z(atoms["H4"], twist_deg)
    order = ["C1", "C2", "H1", "H2", "H3", "H4"]
    return "; ".join(f"{name[0]} {x:.6f} {y:.6f} {z:.6f}" for name in order
                      for x, y, z in [atoms[name]])

def print_geometry(label, geometry):
    print(f"{label}:")
    atoms = geometry.split("; ")
    for i, atom in enumerate(atoms):
        suffix = ";" if i < len(atoms) - 1 else ""
        print(f"  {atom}{suffix}")

# Standard planar equilibrium ethylene geometry (Angstrom)
equilibrium_atoms = {
    "C1": np.array([0.0000,  0.0000,  0.6695]),
    "C2": np.array([0.0000,  0.0000, -0.6695]),
    "H1": np.array([0.0000,  0.9289,  1.2321]),
    "H2": np.array([0.0000, -0.9289,  1.2321]),
    "H3": np.array([0.0000,  0.9289, -1.2321]),
    "H4": np.array([0.0000, -0.9289, -1.2321]),
}


geometry_equilibrium = build_geometry(0.0)
geometry_twisted = build_geometry(90.0)

# print(geometry_equilibrium)
print_geometry("Equilibrium", geometry_equilibrium)
print_geometry("\nTwisted 90", geometry_twisted)

Equilibrium:
  C 0.000000 0.000000 0.669500;
  C 0.000000 0.000000 -0.669500;
  H 0.000000 0.928900 1.232100;
  H 0.000000 -0.928900 1.232100;
  H 0.000000 0.928900 -1.232100;
  H 0.000000 -0.928900 -1.232100

Twisted 90:
  C 0.000000 0.000000 0.669500;
  C 0.000000 0.000000 -0.669500;
  H 0.000000 0.928900 1.232100;
  H 0.000000 -0.928900 1.232100;
  H -0.928900 0.000000 -1.232100;
  H 0.928900 -0.000000 -1.232100


In [ ]:
mol = gto.M(atom=geometry_equilibrium.replace("; ", "\n"), basis="sto-3g",
            charge=0, spin=0, unit="Angstrom")
mf = scf.RHF(mol).run(verbose=0)

# CASCI(2,2): one state-averaged solve gives BOTH ground and S1 -- no separate
# ground-only call needed, since the ground state is already state[0] here.
mc_casci = mcscf.CASCI(mf, 2, 2)
mc_casci.verbose = 0
mc_casci.fcisolver.nstates = 2
e_casci_states = mc_casci.kernel()[0]
casci_ground = e_casci_states[0]
# print(mc_casci.kernel())
# print(e_casci_states  * HARTREE_TO_EV)
casci_excitation_eV = (e_casci_states[1] - e_casci_states[0]) * HARTREE_TO_EV

# CASSCF(2,2): state-averaged the same way, so it now reports an excitation
# energy too, matching CASCI, instead of ground-state energy only.
mc_casscf = mcscf.CASSCF(mf, 2, 2)
mc_casscf = mc_casscf.state_average_([0.5, 0.5])
mc_casscf.verbose = 0
mc_casscf.kernel()
casscf_ground = mc_casscf.e_states[0]
casscf_excitation_eV = (mc_casscf.e_states[1] - mc_casscf.e_states[0]) * HARTREE_TO_EV

# EOM-CCSD: now also reports its ground (CCSD) energy explicitly, and each
# excited root on its own labeled line, matching the other two methods.
mycc = cc.CCSD(mf).run(verbose=0)
eom_excitations_eV = np.array(mycc.eomee_ccsd_singlet(nroots=2)[0]) * HARTREE_TO_EV

print(f"RHF               ground energy: {mf.e_tot:.6f} Ha")
print(f"CASCI(2,2)        ground energy: {casci_ground:.6f} Ha   |  S0->S1 excitation: {casci_excitation_eV:.3f} eV")
print(f"CASSCF(2,2), SA   ground energy: {casscf_ground:.6f} Ha   |  S0->S1 excitation: {casscf_excitation_eV:.3f} eV")
print(f"EOM-CCSD (full)   ground energy: {mycc.e_tot:.6f} Ha   |  S0->S1 excitation: {eom_excitations_eV[0]:.3f} eV")
print(f"                                                          S0->S2 excitation: {eom_excitations_eV[1]:.3f} eV")

(array([-77.11659291, -76.94049593]), array([-1.19817267, -1.0220757 ]), [FCIvector([[-9.68129257e-01, -3.75798580e-15],
           [-3.75798580e-15,  2.50451077e-01]]), FCIvector([[ 9.21674634e-19,  7.07106781e-01],
           [-7.07106781e-01,  3.56277238e-18]])], array([[ 7.01855099e-01, -7.01437104e-01, -1.77650189e-01,
         1.36726387e-01, -8.57149547e-17, -1.49039411e-02,
         7.32570086e-17,  3.61598579e-17, -1.36707143e-16,
        -9.90655278e-15, -1.24208301e-01,  1.68854476e-01,
        -3.12504965e-15,  1.11592777e-01],
       [ 1.99813401e-02, -3.08407101e-02,  4.71423322e-01,
        -4.16088677e-01,  4.42569141e-16,  2.52755645e-02,
        -5.19047265e-16, -3.13939920e-16,  9.19697663e-16,
         6.51679256e-14,  8.20603396e-01, -1.05555235e+00,
         1.83437085e-14, -8.44997183e-01],
       [-1.11305265e-16, -1.69341049e-17,  3.83297416e-17,
         3.37891810e-17, -3.46412399e-15, -5.24708022e-16,
        -2.51825989e-15,  6.35592810e-01,  8.09877497e-01